In [11]:
import pandas as pd
import re
import glob
import os

In [14]:
df = pd.read_csv('../../data/interim/matches/teams/team_stats_cleaned0326.csv')

In [15]:
df

,stat,home_value,away_value,match_id,date
0,general_Possession %,58.30,41.70,763ad58d,Saturday March 30 2024
1,general_Shots,12.00,17.00,763ad58d,Saturday March 30 2024
2,general_Shots on Goal,2.00,10.00,763ad58d,Saturday March 30 2024
3,general_Blocked Shots,4.00,4.00,763ad58d,Saturday March 30 2024
4,general_Total Passes,606.00,370.00,763ad58d,Saturday March 30 2024
...,...,...,...,...,...
60561,possession_81_85,45.45,54.55,7631e105,Sunday February 22\nLumen Field
60562,possession_86_90,29.17,70.83,7631e105,Sunday February 22\nLumen Field
60563,xg_Total Team XG,1.40,0.60,7631e105,Sunday February 22\nLumen Field
60564,xg_Shots,14.00,7.00,7631e105,Sunday February 22\nLumen Field


In [4]:
### pivot long to wide

dfp = df.copy()

In [5]:
dfp['match_id'] = (
    dfp['date'].astype(str).str.strip() + '_' +
    dfp['home_team'].astype(str).str.strip() + '_' +
    dfp['away_team'].astype(str).str.strip()
).str.replace(' ', '')

KeyError: 'date'

In [5]:
dfp['stat_home'] = dfp['stat'] + '_home'
dfp['stat_away'] = dfp['stat'] + '_away'

dfp

,stat,home_value,away_value,match_id,stat_home,stat_away
0,general_Possession %,68.40,31.60,6afff6a7,general_Possession %_home,general_Possession %_away
1,general_Shots,14.00,4.00,6afff6a7,general_Shots_home,general_Shots_away
2,general_Shots on Target,6.00,1.00,6afff6a7,general_Shots on Target_home,general_Shots on Target_away
3,general_Blocked Shots,4.00,2.00,6afff6a7,general_Blocked Shots_home,general_Blocked Shots_away
4,general_Total Passes,648.00,255.00,6afff6a7,general_Total Passes_home,general_Total Passes_away
...,...,...,...,...,...,...
2547,possession_81_85,45.45,54.55,7631e105,possession_81_85_home,possession_81_85_away
2548,possession_86_90,29.17,70.83,7631e105,possession_86_90_home,possession_86_90_away
2549,xg_Total Team XG,1.40,0.60,7631e105,xg_Total Team XG_home,xg_Total Team XG_away
2550,xg_Shots,14.00,7.00,7631e105,xg_Shots_home,xg_Shots_away


In [9]:
home_pivot = dfp.pivot_table(
    index=['match_id'],
    columns='stat_home',
    values='home_value'
)

away_pivot = dfp.pivot_table(
    index=['match_id'],
    columns='stat_away',
    values='away_value'
)

In [10]:
final = pd.concat([home_pivot, away_pivot], axis=1).reset_index()

In [11]:
final

,match_id,general_Aerial Duels Won_home,general_Blocked Shots_home,general_Blocked_home,general_Clean Sheets_home,general_Clearances_home,general_Corners_home,general_Expected Goals_home,general_Fouls_home,general_Goalkeeper Saves_home,...,possession_76_80_away,possession_81_85_away,possession_86_90_away,shooting_Blocked_away,shooting_Goals_away,shooting_Off Target_away,shooting_On Target_away,xg_Shots On Target_away,xg_Shots_away,xg_Total Team XG_away
0,08e147bc,7.0,8.0,8.0,0.0,4.0,7.0,2.7,11.0,2.0,...,51.90,51.80,70.78,2.0,0.0,4.0,3.0,3.0,8.0,0.9
1,12769fea,12.0,4.0,4.0,0.0,7.0,6.0,1.1,15.0,2.0,...,42.50,43.39,45.78,2.0,0.0,2.0,3.0,3.0,7.0,0.5
2,1b022c5c,14.0,2.0,2.0,0.0,7.0,5.0,2.5,12.0,2.0,...,32.02,12.67,43.26,3.0,0.0,1.0,2.0,2.0,6.0,0.4
3,2046cddd,14.0,2.0,2.0,0.0,5.0,1.0,1.4,19.0,3.0,...,59.10,75.00,38.17,5.0,0.0,3.0,3.0,3.0,13.0,1.1
4,245c0550,5.0,4.0,4.0,0.0,2.0,3.0,3.4,16.0,0.0,...,67.88,78.89,39.93,0.0,1.0,2.0,1.0,1.0,4.0,0.8
5,2f937f25,10.0,7.0,7.0,0.0,7.0,4.0,0.9,6.0,4.0,...,90.73,65.57,66.05,4.0,2.0,7.0,6.0,6.0,18.0,1.5
6,314976c2,13.0,2.0,2.0,0.0,2.0,7.0,1.5,8.0,2.0,...,41.38,34.93,26.65,3.0,2.0,4.0,4.0,4.0,11.0,1.8
7,384d63a9,7.0,7.0,7.0,0.0,9.0,4.0,3.7,14.0,3.0,...,65.04,70.89,68.26,6.0,0.0,3.0,3.0,3.0,13.0,0.9
8,3f3f51ea,26.0,7.0,7.0,0.0,6.0,15.0,3.9,16.0,1.0,...,66.93,45.81,23.36,1.0,0.0,3.0,1.0,1.0,7.0,0.5
9,5ba741bf,10.0,4.0,4.0,0.0,10.0,4.0,1.0,13.0,5.0,...,55.17,54.00,32.18,3.0,0.0,5.0,5.0,5.0,15.0,1.6


In [12]:
### simplify colnames
final.columns = [c.replace(' ', '_').replace('%', 'pct').replace('-', '_').lower() for c in final.columns]

final

,match_id,general_aerial_duels_won_home,general_blocked_shots_home,general_blocked_home,general_clean_sheets_home,general_clearances_home,general_corners_home,general_expected_goals_home,general_fouls_home,general_goalkeeper_saves_home,...,possession_76_80_away,possession_81_85_away,possession_86_90_away,shooting_blocked_away,shooting_goals_away,shooting_off_target_away,shooting_on_target_away,xg_shots_on_target_away,xg_shots_away,xg_total_team_xg_away
0,08e147bc,7.0,8.0,8.0,0.0,4.0,7.0,2.7,11.0,2.0,...,51.90,51.80,70.78,2.0,0.0,4.0,3.0,3.0,8.0,0.9
1,12769fea,12.0,4.0,4.0,0.0,7.0,6.0,1.1,15.0,2.0,...,42.50,43.39,45.78,2.0,0.0,2.0,3.0,3.0,7.0,0.5
2,1b022c5c,14.0,2.0,2.0,0.0,7.0,5.0,2.5,12.0,2.0,...,32.02,12.67,43.26,3.0,0.0,1.0,2.0,2.0,6.0,0.4
3,2046cddd,14.0,2.0,2.0,0.0,5.0,1.0,1.4,19.0,3.0,...,59.10,75.00,38.17,5.0,0.0,3.0,3.0,3.0,13.0,1.1
4,245c0550,5.0,4.0,4.0,0.0,2.0,3.0,3.4,16.0,0.0,...,67.88,78.89,39.93,0.0,1.0,2.0,1.0,1.0,4.0,0.8
5,2f937f25,10.0,7.0,7.0,0.0,7.0,4.0,0.9,6.0,4.0,...,90.73,65.57,66.05,4.0,2.0,7.0,6.0,6.0,18.0,1.5
6,314976c2,13.0,2.0,2.0,0.0,2.0,7.0,1.5,8.0,2.0,...,41.38,34.93,26.65,3.0,2.0,4.0,4.0,4.0,11.0,1.8
7,384d63a9,7.0,7.0,7.0,0.0,9.0,4.0,3.7,14.0,3.0,...,65.04,70.89,68.26,6.0,0.0,3.0,3.0,3.0,13.0,0.9
8,3f3f51ea,26.0,7.0,7.0,0.0,6.0,15.0,3.9,16.0,1.0,...,66.93,45.81,23.36,1.0,0.0,3.0,1.0,1.0,7.0,0.5
9,5ba741bf,10.0,4.0,4.0,0.0,10.0,4.0,1.0,13.0,5.0,...,55.17,54.00,32.18,3.0,0.0,5.0,5.0,5.0,15.0,1.6


In [16]:
def reframe_stats(df):
    # Normalise inconsistent stat names before processing
    df = df.copy()
    df['stat'] = df['stat'].replace({
        'general_Shots on Target': 'general_Shots on Goal',
        'xg_Shots On Target':      'xg_Shots on Goal'
    })
    
    processed_matches = []
    
    for match_id, match_df in df.groupby('match_id'):
        match_record = {'match_id': match_id}
        
        for col in match_df.columns:
            if col not in ['match_id', 'stat', 'home_value', 'away_value']:
                match_record[col] = match_df[col].iloc[0]
        
        for _, row in match_df.iterrows():
            if pd.notna(row['stat']):
                stat_clean = row['stat'].replace(' ', '_').replace('%', 'pct').replace('-', '_').lower()
                match_record[f"{stat_clean}_home"] = row['home_value']
                match_record[f"{stat_clean}_away"] = row['away_value']
        
        processed_matches.append(match_record)
    
    return pd.DataFrame(processed_matches)

In [4]:
import glob
import os

In [20]:
for file in glob.glob('*.csv'):
    filename = os.path.basename(file)
    df = pd.read_csv(file)
    cleaned_df = reframe_stats(filename, df)
    cleaned_df.to_csv(
        f"G:/My Drive/GitHubProjects/MLS/data/data_clean/matches/raw/reframed_stats/{filename}",
        index=False)

In [53]:
df = pd.read_csv('../../data/interim/matches/teams/team_stats_cleaned0326_post_26.csv')


In [54]:
df

,stat,home_value,away_value,match_id,date
0,general_Possession %,68.40,31.60,6afff6a7,2026-03-07
1,general_Shots,14.00,4.00,6afff6a7,2026-03-07
2,general_Shots on Target,6.00,1.00,6afff6a7,2026-03-07
3,general_Blocked Shots,4.00,2.00,6afff6a7,2026-03-07
4,general_Total Passes,648.00,255.00,6afff6a7,2026-03-07
...,...,...,...,...,...
2547,possession_81_85,45.45,54.55,7631e105,2026-02-22
2548,possession_86_90,29.17,70.83,7631e105,2026-02-22
2549,xg_Total Team XG,1.40,0.60,7631e105,2026-02-22
2550,xg_Shots,14.00,7.00,7631e105,2026-02-22


In [56]:
wide_df = reframe_stats(df)



In [57]:
wide_df

,match_id,date,general_possession_pct_home,general_possession_pct_away,general_shots_home,general_shots_away,general_shots_on_goal_home,general_shots_on_goal_away,general_blocked_shots_home,general_blocked_shots_away,...,possession_81_85_home,possession_81_85_away,possession_86_90_home,possession_86_90_away,xg_total_team_xg_home,xg_total_team_xg_away,xg_shots_home,xg_shots_away,xg_shots_on_goal_home,xg_shots_on_goal_away
0,08e147bc,2026-03-07,52.0,48.0,14.0,8.0,5.0,3.0,8.0,2.0,...,48.20,51.80,29.22,70.78,2.7,0.9,14.0,8.0,5.0,3.0
1,12769fea,2026-02-28,61.4,38.6,11.0,7.0,4.0,3.0,4.0,2.0,...,56.61,43.39,54.22,45.78,1.0,0.5,11.0,7.0,4.0,3.0
2,1b022c5c,2026-02-21,64.1,35.9,12.0,6.0,8.0,2.0,2.0,3.0,...,87.33,12.67,56.74,43.26,2.5,0.4,12.0,6.0,8.0,2.0
3,2046cddd,2026-02-21,50.4,49.6,7.0,13.0,1.0,3.0,2.0,5.0,...,25.00,75.00,61.83,38.17,1.4,1.1,7.0,13.0,1.0,3.0
4,245c0550,2026-02-21,52.3,47.7,14.0,4.0,8.0,1.0,4.0,0.0,...,21.11,78.89,60.07,39.93,3.4,0.8,14.0,4.0,8.0,1.0
5,2f937f25,2026-02-28,35.8,64.2,12.0,18.0,2.0,6.0,7.0,4.0,...,34.43,65.57,33.95,66.05,0.9,1.5,12.0,18.0,2.0,6.0
6,314976c2,2026-02-21,54.4,45.6,10.0,11.0,6.0,4.0,2.0,3.0,...,65.07,34.93,73.35,26.65,1.5,1.8,10.0,11.0,6.0,4.0
7,384d63a9,2026-02-21,34.5,65.5,16.0,13.0,6.0,3.0,7.0,6.0,...,29.11,70.89,31.74,68.26,3.7,0.9,16.0,13.0,6.0,3.0
8,3f3f51ea,2026-02-21,42.0,58.0,18.0,7.0,6.0,1.0,7.0,1.0,...,54.19,45.81,76.64,23.36,3.9,0.5,18.0,7.0,6.0,1.0
9,5ba741bf,2026-03-07,60.8,39.2,10.0,15.0,6.0,5.0,4.0,3.0,...,46.00,54.00,67.82,32.18,1.0,1.6,10.0,15.0,6.0,5.0


In [58]:
## find any na values
nasum = wide_df.isna().sum()

In [59]:
nasum

match_id                       0
date                           0
general_possession_pct_home    0
general_possession_pct_away    0
general_shots_home             0
                              ..
xg_total_team_xg_away          0
xg_shots_home                  0
xg_shots_away                  0
xg_shots_on_goal_home          0
xg_shots_on_goal_away          0
Length: 118, dtype: int64

In [60]:
wide_df.to_csv('../../data/interim/matches/teams/mls_match_team_stats_reframed0326_post26.csv', index=False)

In [49]:
wide = wide_df.copy()

In [62]:
wide_df

,match_id,date,general_possession_pct_home,general_possession_pct_away,general_shots_home,general_shots_away,general_shots_on_goal_home,general_shots_on_goal_away,general_blocked_shots_home,general_blocked_shots_away,...,possession_81_85_home,possession_81_85_away,possession_86_90_home,possession_86_90_away,xg_total_team_xg_home,xg_total_team_xg_away,xg_shots_home,xg_shots_away,xg_shots_on_goal_home,xg_shots_on_goal_away
0,08e147bc,2026-03-07,52.0,48.0,14.0,8.0,5.0,3.0,8.0,2.0,...,48.20,51.80,29.22,70.78,2.7,0.9,14.0,8.0,5.0,3.0
1,12769fea,2026-02-28,61.4,38.6,11.0,7.0,4.0,3.0,4.0,2.0,...,56.61,43.39,54.22,45.78,1.0,0.5,11.0,7.0,4.0,3.0
2,1b022c5c,2026-02-21,64.1,35.9,12.0,6.0,8.0,2.0,2.0,3.0,...,87.33,12.67,56.74,43.26,2.5,0.4,12.0,6.0,8.0,2.0
3,2046cddd,2026-02-21,50.4,49.6,7.0,13.0,1.0,3.0,2.0,5.0,...,25.00,75.00,61.83,38.17,1.4,1.1,7.0,13.0,1.0,3.0
4,245c0550,2026-02-21,52.3,47.7,14.0,4.0,8.0,1.0,4.0,0.0,...,21.11,78.89,60.07,39.93,3.4,0.8,14.0,4.0,8.0,1.0
5,2f937f25,2026-02-28,35.8,64.2,12.0,18.0,2.0,6.0,7.0,4.0,...,34.43,65.57,33.95,66.05,0.9,1.5,12.0,18.0,2.0,6.0
6,314976c2,2026-02-21,54.4,45.6,10.0,11.0,6.0,4.0,2.0,3.0,...,65.07,34.93,73.35,26.65,1.5,1.8,10.0,11.0,6.0,4.0
7,384d63a9,2026-02-21,34.5,65.5,16.0,13.0,6.0,3.0,7.0,6.0,...,29.11,70.89,31.74,68.26,3.7,0.9,16.0,13.0,6.0,3.0
8,3f3f51ea,2026-02-21,42.0,58.0,18.0,7.0,6.0,1.0,7.0,1.0,...,54.19,45.81,76.64,23.36,3.9,0.5,18.0,7.0,6.0,1.0
9,5ba741bf,2026-03-07,60.8,39.2,10.0,15.0,6.0,5.0,4.0,3.0,...,46.00,54.00,67.82,32.18,1.0,1.6,10.0,15.0,6.0,5.0


In [68]:
old_wide = pd.read_csv('../../data/interim/matches/teams/mls_match_team_stats_with_match_id.csv')


new = pd.concat([old_wide, wide_df], axis=0, ignore_index=True)

In [69]:
new = new.drop(columns=['date', 'home_team', 'away_team', 'xg_shots_on_target_home', 'xg_shots_on_target_away', 'xg_shots_on_goal_away', 'xg_shots_on_goal_home'])

In [70]:
new.isna().sum()

general_aerial_duels_won_home    10
general_blocked_shots_home       10
general_blocked_home             10
general_clean_sheets_home        10
general_clearances_home          10
                                 ..
shooting_off_target_away          9
shooting_on_target_away           9
xg_shots_away                     9
xg_total_team_xg_away            10
match_id                          0
Length: 115, dtype: int64

In [71]:
### move match id to front

new = new[['match_id'] + [col for col in new.columns if col != 'match_id']]

In [72]:
new

,match_id,general_aerial_duels_won_home,general_blocked_shots_home,general_blocked_home,general_clean_sheets_home,general_clearances_home,general_corners_home,general_expected_goals_home,general_fouls_home,general_goalkeeper_saves_home,...,possession_71_75_away,possession_76_80_away,possession_81_85_away,possession_86_90_away,shooting_blocked_away,shooting_goals_away,shooting_off_target_away,shooting_on_target_away,xg_shots_away,xg_total_team_xg_away
0,43ba7695,6.0,4.0,4.0,0.0,6.0,9.0,3.0,12.0,3.0,...,50.28,61.16,43.51,34.91,2.0,3.0,2.0,6.0,10.0,2.0
1,85bb856a,14.0,1.0,1.0,0.0,3.0,2.0,2.0,16.0,1.0,...,57.37,40.05,74.32,41.92,3.0,4.0,1.0,5.0,9.0,1.7
2,588d481f,8.0,4.0,4.0,0.0,3.0,4.0,1.4,6.0,3.0,...,47.48,35.72,16.70,55.20,1.0,1.0,3.0,4.0,8.0,0.4
3,1b458adf,8.0,9.0,9.0,0.0,8.0,3.0,1.1,10.0,2.0,...,33.44,42.86,23.66,32.48,3.0,3.0,2.0,5.0,11.0,1.5
4,5bce4433,10.0,2.0,2.0,0.0,5.0,2.0,1.3,12.0,4.0,...,41.31,45.27,33.57,42.54,3.0,0.0,1.0,4.0,9.0,0.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1040,dee239a9,6.0,1.0,1.0,0.0,16.0,6.0,1.3,5.0,3.0,...,51.39,40.69,78.63,34.66,6.0,2.0,5.0,5.0,16.0,1.1
1041,eaff4742,10.0,5.0,5.0,0.0,10.0,12.0,2.6,12.0,3.0,...,40.48,46.65,32.17,15.81,3.0,0.0,6.0,3.0,12.0,1.2
1042,f5d62f89,10.0,4.0,4.0,0.0,12.0,10.0,1.8,13.0,2.0,...,45.98,46.59,54.37,60.45,1.0,0.0,3.0,2.0,8.0,0.4
1043,fc3f246b,32.0,2.0,2.0,0.0,3.0,11.0,1.4,20.0,2.0,...,46.79,52.76,47.75,40.09,2.0,1.0,2.0,3.0,7.0,0.8


In [50]:
df = wide.copy()

In [51]:
month_map = {
        "january": "01", "february": "02", "march": "03", "april": "04",
        "may": "05", "june": "06", "july": "07", "august": "08",
        "september": "09", "october": "10", "november": "11", "december": "12"
    }

days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']


df['date'] = df['date'].str.split('\n').str[0]

df['date'] = df['date'].apply(lambda x: ' '.join([word for word in x.split() if word not in days]))


df['date'] = df['date'].apply(
        lambda x: re.sub(
            r"([A-Za-z]+)",
            lambda m: month_map[m.group(1).lower()],
            x
        )
    )

df['date'] = df['date'].apply(lambda x: x + " 2026" if re.search(r"\d{4}$", x) is None else x)

df['date'] = pd.to_datetime(df['date'], format="%m %d %Y")


In [52]:
wide = df.copy()

wide

,match_id,date,general_possession_pct_home,general_possession_pct_away,general_shots_home,general_shots_away,general_shots_on_goal_home,general_shots_on_goal_away,general_blocked_shots_home,general_blocked_shots_away,...,possession_81_85_home,possession_81_85_away,possession_86_90_home,possession_86_90_away,xg_total_team_xg_home,xg_total_team_xg_away,xg_shots_home,xg_shots_away,xg_shots_on_goal_home,xg_shots_on_goal_away
0,00093bf5,2026-08-16,50.7,49.3,19.0,13.0,5.0,3.0,7.0,3.0,...,40.34,59.66,75.20,24.80,2.5,1.5,19.0,13.0,5.0,3.0
1,006b3126,2026-10-11,57.1,42.9,20.0,10.0,10.0,3.0,5.0,2.0,...,71.18,28.82,73.89,26.11,2.8,1.3,20.0,10.0,10.0,3.0
2,00ba135d,2024-05-04,56.7,43.3,16.0,15.0,7.0,5.0,4.0,3.0,...,65.14,34.86,61.36,38.64,3.0,1.5,16.0,15.0,7.0,5.0
3,00cdab1a,2026-06-14,56.2,43.8,20.0,8.0,7.0,1.0,8.0,4.0,...,50.91,49.09,53.94,46.06,1.8,0.8,20.0,8.0,7.0,1.0
4,0182f52b,2026-07-19,51.5,48.5,17.0,10.0,4.0,4.0,3.0,3.0,...,45.08,54.92,67.50,32.50,0.9,2.0,17.0,10.0,4.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1041,ff30d11e,2024-03-09,59.2,40.8,8.0,11.0,5.0,6.0,3.0,2.0,...,58.25,41.75,76.89,23.11,1.3,1.6,8.0,11.0,5.0,6.0
1042,ff46f6c3,2024-06-22,49.6,50.4,9.0,16.0,3.0,8.0,1.0,3.0,...,60.43,39.57,45.05,54.95,1.7,2.6,9.0,16.0,3.0,8.0
1043,ff4ea1b4,2026-05-17,44.7,55.3,6.0,11.0,3.0,4.0,0.0,3.0,...,47.27,52.73,62.95,37.05,0.9,0.4,6.0,11.0,3.0,4.0
1044,ffb5ce29,2024-07-13,62.2,37.8,17.0,9.0,7.0,0.0,4.0,7.0,...,48.85,51.15,44.12,55.88,2.0,0.6,17.0,9.0,7.0,0.0


In [73]:
new.to_csv('../../data/interim/matches/teams/mls_match_team_stats_reframed_full_until0326.csv', index=False)